# Segmentação semântica — Oxford-IIIT Pet + UNet

Pipeline: dataset → treino → métricas (IoU, acurácia) → visualização → TFLite.

**Recomendado:** Runtime → Change runtime type → **GPU (T4)**.

## Dependências

In [ ]:
!pip install -q segmentation-models-pytorch torchmetrics tqdm

## Configuração

> Se ainda aparecer erro de `numpy.dtype size changed`, use **Runtime → Restart session** e execute novamente a partir desta célula (a célula de dependências não precisa ser repetida).

In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from tqdm import tqdm

import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = Path("/content/data")
IMG_SIZE = 128
NUM_CLASSES = 2  # 0=fundo, 1=pet
BATCH_SIZE = 8
EPOCHS = 10
LR = 1e-3
SUBSET_SIZE = 1000  # recorte reduzido do dataset
VAL_RATIO = 0.2
IGNORE_INDEX = 255
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")

## Preparo dos dados (Oxford-IIIT Pet)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

mask_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
])


def trimap_to_mask(trimap: torch.Tensor) -> torch.Tensor:
    """Converte trimap (1=pet, 2=fundo, 3=ignorar) em máscara binária."""
    mask = torch.full_like(trimap, IGNORE_INDEX, dtype=torch.long)
    mask[trimap == 1] = 1  # pet
    mask[trimap == 2] = 0  # fundo
    return mask


class PetSegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        image, trimap = self.base[idx]
        mask = mask_transform(trimap)
        if isinstance(mask, torch.Tensor):
            mask = mask.squeeze(0).long()
        else:
            mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        mask = trimap_to_mask(mask)
        image = image_transform(image)
        return image, mask


base_dataset = OxfordIIITPet(
    root=str(DATA_ROOT),
    download=True,
    target_types="segmentation",
)

indices = torch.randperm(len(base_dataset), generator=torch.Generator().manual_seed(SEED))
subset_indices = indices[:SUBSET_SIZE].tolist()
subset = Subset(base_dataset, subset_indices)
dataset = PetSegmentationDataset(subset)

val_size = int(len(dataset) * VAL_RATIO)
train_size = len(dataset) - val_size
train_set, val_set = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Total (subset): {len(dataset)} | Treino: {train_size} | Validação: {val_size}")

## Construção do modelo (UNet)

In [ ]:
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

model.summary() if hasattr(model, "summary") else print(model)

## Treinamento

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_acc": []}


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    iou_metric = MulticlassJaccardIndex(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX).to(DEVICE)
    acc_metric = MulticlassAccuracy(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX).to(DEVICE)

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, masks in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            logits = model(images)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            iou_metric.update(preds, masks)
            acc_metric.update(preds, masks)

    n = len(loader.dataset)
    return (
        total_loss / n,
        iou_metric.compute().item(),
        acc_metric.compute().item(),
    )


for epoch in range(EPOCHS):
    train_loss, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_iou, val_acc = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    history["val_acc"].append(val_acc)

    print(
        f"Época {epoch + 1}/{EPOCHS} | "
        f"loss treino: {train_loss:.4f} | loss val: {val_loss:.4f} | "
        f"IoU: {val_iou:.4f} | acurácia: {val_acc:.4f}"
    )

## Avaliação final

In [ ]:
print(f"IoU (validação): {history['val_iou'][-1]:.4f}")
print(f"Acurácia (validação): {history['val_acc'][-1]:.4f}")

## Visualização (imagem, GT, predição, overlay)

In [ ]:
def denormalize(tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    return img.permute(1, 2, 0).numpy().clip(0, 1)


def overlay_mask(image: np.ndarray, mask: np.ndarray, color=(1.0, 0.0, 0.0), alpha=0.45) -> np.ndarray:
    out = image.copy()
    fg = mask.astype(bool)
    out[fg] = out[fg] * (1 - alpha) + np.array(color) * alpha
    return out


model.eval()
images, masks = next(iter(val_loader))
with torch.no_grad():
    preds = model(images.to(DEVICE)).argmax(dim=1).cpu()

n_show = min(4, len(images))
fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.5 * n_show))
if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

for i in range(n_show):
    img = denormalize(images[i])
    gt = masks[i].numpy()
    pred = preds[i].numpy()
    gt_vis = np.where(gt == IGNORE_INDEX, 0, gt)
    pred_vis = pred

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Imagem")
    axes[i, 1].imshow(gt_vis, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("Máscara GT")
    axes[i, 2].imshow(pred_vis, cmap="gray", vmin=0, vmax=1)
    axes[i, 2].set_title("Predição")
    axes[i, 3].imshow(overlay_mask(img, pred_vis == 1))
    axes[i, 3].set_title("Overlay")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.savefig("segmentation_results.png", dpi=120, bbox_inches="tight")
plt.show()

## Exportação para TFLite (FP32 e INT8)

> O TensorFlow é instalado **somente na célula seguinte** para não quebrar o NumPy/torchvision das células anteriores.

In [ ]:
!pip install -q onnx onnxsim onnx2tf tensorflow
!pip install -q --upgrade --force-reinstall numpy

In [ ]:
import onnx
import onnxsim
from onnx2tf import convert


class ExportModel(nn.Module):
    """Entrada em [0, 1] (NCHW) → logits (NCHW) para simplificar o app Android."""

    def __init__(self, net):
        super().__init__()
        self.net = net
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.net(x)


export_model = ExportModel(model).eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = "unet.onnx"

torch.onnx.export(
    export_model,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=18,
    dynamic_axes=None,
)

onnx_model = onnx.load(onnx_path)
onnx_model, ok = onnxsim.simplify(onnx_model)
assert ok, "Falha ao simplificar ONNX"
onnx.save(onnx_model, onnx_path)

saved_model_dir = "saved_model"
if os.path.exists(saved_model_dir):
    shutil.rmtree(saved_model_dir)

# onnx2tf recente gera .tflite diretamente (sem SavedModel clássico)
convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=saved_model_dir,
    copy_onnx_input_output_names_to_tflite=True,
)

output_dir = Path(saved_model_dir)
float32_path = output_dir / "unet_float32.tflite"
float16_path = output_dir / "unet_float16.tflite"

if not float32_path.exists():
    candidates = sorted(output_dir.glob("*float32*.tflite"))
    if not candidates:
        raise FileNotFoundError(
            f"Nenhum .tflite float32 encontrado em {output_dir}. "
            f"Arquivos: {[p.name for p in output_dir.glob('*')]}"
        )
    float32_path = candidates[0]

shutil.copy(float32_path, "model.tflite")
print(f"model.tflite salvo ({os.path.getsize('model.tflite') / 1024:.1f} KB)")

if float16_path.exists():
    shutil.copy(float16_path, "model_float16.tflite")
    print(f"model_float16.tflite salvo ({os.path.getsize('model_float16.tflite') / 1024:.1f} KB)")

In [ ]:
import tensorflow as tf


def build_calibration_sample(img_tensor: torch.Tensor) -> np.ndarray:
    """Converte batch ImageNet-normalizado → entrada [0,1] NCHW para calibração."""
    x = img_tensor.unsqueeze(0).cpu().numpy().astype(np.float32)
    mean = np.array(IMAGENET_MEAN, dtype=np.float32).reshape(1, 3, 1, 1)
    std = np.array(IMAGENET_STD, dtype=np.float32).reshape(1, 3, 1, 1)
    return x * std + mean


def representative_dataset():
    count = 0
    for images, _ in train_loader:
        for img in images:
            if count >= 100:
                return
            yield [build_calibration_sample(img)]
            count += 1


# Quantização INT8 (opcional) a partir do .tflite FP32 gerado pelo onnx2tf
try:
    if not hasattr(tf.lite.TFLiteConverter, "from_file"):
        raise AttributeError("TFLiteConverter.from_file não disponível nesta versão do TensorFlow")

    converter = tf.lite.TFLiteConverter.from_file(str(float32_path))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_int8 = converter.convert()
    with open("model_int8.tflite", "wb") as f:
        f.write(tflite_int8)
    print(f"model_int8.tflite salvo ({len(tflite_int8) / 1024:.1f} KB)")
except Exception as exc:
    print(f"Quantização INT8 indisponível: {exc}")
    print("Use model.tflite (FP32) ou model_float16.tflite no Android.")

torch.save(model.state_dict(), "unet_pytorch.pt")

android_assets = Path("../Android/app/src/main/assets")
if android_assets.parent.exists():
    android_assets.mkdir(parents=True, exist_ok=True)
    shutil.copy("model.tflite", android_assets / "model.tflite")
    print(f"Modelo copiado para {android_assets.resolve()}")
else:
    print("Copie manualmente model.tflite para Android/app/src/main/assets/")

## Comparação de tamanhos

In [ ]:
sizes = {
    "PyTorch (.pt)": os.path.getsize("unet_pytorch.pt"),
    "TFLite FP32": os.path.getsize("model.tflite"),
}
if os.path.exists("model_float16.tflite"):
    sizes["TFLite FP16"] = os.path.getsize("model_float16.tflite")
if os.path.exists("model_int8.tflite"):
    sizes["TFLite INT8"] = os.path.getsize("model_int8.tflite")

for name, size in sizes.items():
    print(f"{name}: {size / 1024:.1f} KB")